# Train FOMO (người / xe) cho Smart Street Lighting — chạy trên Kaggle

**Mục tiêu:** train một model kiểu **FOMO** (Faster Objects, More Objects) nhận diện `car` / `human` trên ảnh grayscale,
lượng tử hoá **int8** và xuất ra `.tflite` + file C (`.h/.cc`) để chạy bằng **TensorFlow Lite Micro trên ESP32-S3**.

**Vì sao là FOMO chứ không phải YOLO:** hệ thống chỉ cần biết *vật ở zone nào*, nên chỉ cần **tâm vật thể**.
FOMO bỏ phần dự đoán bounding box và chỉ dự đoán một **heatmap trên lưới** (mỗi ô: nền / car / human),
nhờ đó model chỉ còn vài trăm KB và chạy được trên MCU.

```
Ảnh 160×160 gray ─► MobileNetV2 α=0.35 (cắt ở block_6_expand_relu, stride 8) ─► Conv1×1(32) ─► Conv1×1(3)+softmax
                                                                                     │
                                                       heatmap 20×20×3 (nền, car, human)
                                                                                     ▼
                                     ngưỡng + gom ô liền nhau ─► danh sách tâm {class, x, y, score}
```

## Chuẩn bị trên Kaggle
1. **Upload dataset:** *Create → New Dataset* → upload `HTN.v2i.coco.zip` (export COCO từ Roboflow).
2. Trong notebook: **Add Input** → chọn dataset vừa tạo. Notebook **tự tìm** `train/_annotations.coco.json` trong `/kaggle/input`.
3. **Settings → Accelerator: GPU (T4 hoặc P100)**, **Internet: On** (để tải trọng số ImageNet của MobileNetV2).
4. *Run All*. Kết quả nằm trong `/kaggle/working/fomo_output/` (tab **Output** để tải về).

## Thứ tự các phần
1. Cấu hình & nạp dữ liệu COCO → 2. Khám phá dữ liệu → 3. Tạo nhãn FOMO (lưới) → 4. Augmentation →
5. Model → 6. Loss có trọng số → 7. Metric theo tâm (P/R/F1) → 8. Train → 9. Chọn ngưỡng trên **val**, đánh giá trên **test** →
10. Lượng tử hoá int8 & so sánh với float → 11. Tài nguyên (Flash/RAM/op) → 12. Xuất file cho firmware

## 1. Import & kiểm tra môi trường

In [ ]:
import glob
import json
import shutil
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from scipy import ndimage
from tensorflow import keras

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE (training will be slow — enable GPU in Settings)")

## 2. Cấu hình

Các tham số quan trọng (đã cân nhắc theo dataset và board):

| Tham số | Giá trị | Lý do |
|---|---|---|
| `IMG_SIZE` | 160 | Ảnh gốc 320×240, vật trung vị ~24×20 px → còn ~12×13 px sau khi thu nhỏ (≥ 8–10 px như kế hoạch yêu cầu) |
| `GRID_STRIDE` | 8 | Cắt MobileNetV2 ở `block_6_expand_relu` → lưới **20×20**, mỗi ô 8 px. Đo trên dataset: chỉ 0.4% vật bị gộp chung ô (lưới 10×10 gộp ~5%) |
| `ALPHA` | 0.35 | Bản MobileNetV2 nhỏ nhất có trọng số ImageNet, giống FOMO của Edge Impulse |
| `OBJECT_WEIGHT` | 100 | Lưới 400 ô mà mỗi ảnh chỉ có ~3 ô có vật → phải tăng trọng số ô có vật, nếu không model học "đoán toàn nền" |
| `MATCH_DIST_CELLS` | 1.5 | Dự đoán được tính là đúng nếu tâm cách tâm thật ≤ 1.5 ô (12 px trên ảnh 160) |

> **Lưu ý resize:** notebook **ép (squash)** cả ảnh về 160×160, không giữ tỉ lệ. Firmware phải làm **y hệt**
> (cắt ROI rồi resize bilinear về 160×160), nếu không ảnh lúc chạy thật sẽ khác ảnh lúc train.

In [ ]:
SEED = 42

# --- Model geometry ---
IMG_SIZE = 160                         # square model input (grayscale)
GRID_STRIDE = 8                        # MobileNetV2 cut at block_6_expand_relu -> stride 8
GRID_SIZE = IMG_SIZE // GRID_STRIDE    # 20x20 output heatmap
ALPHA = 0.35                           # MobileNetV2 width multiplier
CUT_LAYER = "block_6_expand_relu"
HEAD_FILTERS = 32

# --- Training ---
BATCH_SIZE = 32
EPOCHS_FROZEN = 5          # phase 1: backbone frozen, only the head learns
EPOCHS_FINETUNE = 55       # phase 2: whole network fine-tuned
LR_FROZEN = 1e-3
LR_FINETUNE = 5e-4
OBJECT_WEIGHT = 100.0      # loss weight of object cells vs background cells
USE_VERTICAL_FLIP = False  # only enable if the camera looks straight down (top-down view)

# --- Evaluation ---
MATCH_DIST_CELLS = 1.5     # max centroid distance (in grid cells) to count a true positive
DEFAULT_THRESHOLD = 0.5    # used during training to track val F1
THRESHOLDS = np.round(np.arange(0.30, 0.951, 0.05), 2)

# --- Quantization ---
NUM_CALIB_IMAGES = 300     # representative images, taken from TRAIN only

OUTPUT_DIR = Path("/kaggle/working/fomo_output") if Path("/kaggle/working").exists() else Path("fomo_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

keras.utils.set_random_seed(SEED)
print("Grid:", GRID_SIZE, "x", GRID_SIZE, "| output dir:", OUTPUT_DIR)

## 3. Tìm và nạp dataset COCO

Roboflow export COCO có 3 thư mục `train/`, `valid/`, `test/`, mỗi thư mục có `_annotations.coco.json`.
Roboflow thêm một category cha (`HTN`, `supercategory = "none"`) không gán cho box nào, nên bỏ nó đi.
Class index trong model: **0 = nền**, 1..C = các class thật (theo thứ tự id trong COCO).

In [ ]:
def find_coco_root() -> Path:
    # Return the directory that contains train/ valid/ test/ COCO splits.
    search_roots = ["/kaggle/input", "."]
    for root in search_roots:
        hits = sorted(glob.glob(f"{root}/**/train/_annotations.coco.json", recursive=True))
        if hits:
            if len(hits) > 1:
                print("WARNING: several COCO datasets found, using the first one:\n  " + "\n  ".join(hits))
            return Path(hits[0]).parent.parent
    raise FileNotFoundError(
        "No train/_annotations.coco.json found. Add the Roboflow COCO export as a Kaggle input dataset."
    )


DATA_ROOT = find_coco_root()
SPLIT_DIRS = {"train": DATA_ROOT / "train", "val": DATA_ROOT / "valid", "test": DATA_ROOT / "test"}
print("Dataset root:", DATA_ROOT)

In [ ]:
@dataclass(frozen=True)
class Sample:
    # One annotated image. Boxes are in ORIGINAL image pixels (x, y, w, h).
    path: Path
    width: int
    height: int
    boxes: np.ndarray   # shape (N, 4), float32
    labels: np.ndarray  # shape (N,), int, values in 1..num_classes


def read_class_names(split_dir: Path) -> tuple[list[str], dict[int, int]]:
    # Return class names and a mapping COCO category id -> model class index (1..C).
    with open(split_dir / "_annotations.coco.json", encoding="utf-8") as f:
        coco = json.load(f)
    cats = sorted((c for c in coco["categories"] if c.get("supercategory") != "none"), key=lambda c: c["id"])
    if not cats:
        raise ValueError("No usable categories found in COCO file")
    names = [c["name"] for c in cats]
    cat_to_idx = {c["id"]: i + 1 for i, c in enumerate(cats)}
    return names, cat_to_idx


def load_coco_split(split_dir: Path, cat_to_idx: dict[int, int]) -> list[Sample]:
    # Load all images of one split together with their boxes.
    with open(split_dir / "_annotations.coco.json", encoding="utf-8") as f:
        coco = json.load(f)

    anns_by_image: dict[int, list[dict]] = {}
    for ann in coco["annotations"]:
        if ann["category_id"] not in cat_to_idx:
            raise ValueError(f"Annotation {ann['id']} uses unknown category {ann['category_id']}")
        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue  # degenerate box, skip it
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    samples = []
    for img in coco["images"]:
        anns = anns_by_image.get(img["id"], [])
        boxes = np.array([a["bbox"] for a in anns], dtype=np.float32).reshape(-1, 4)
        labels = np.array([cat_to_idx[a["category_id"]] for a in anns], dtype=np.int64)
        samples.append(Sample(split_dir / img["file_name"], img["width"], img["height"], boxes, labels))
    return samples


CLASS_NAMES, CAT_TO_IDX = read_class_names(SPLIT_DIRS["train"])
NUM_CLASSES = len(CLASS_NAMES)
ALL_NAMES = ["background"] + CLASS_NAMES

samples = {split: load_coco_split(d, CAT_TO_IDX) for split, d in SPLIT_DIRS.items()}
for split in ("val", "test"):
    names, _ = read_class_names(SPLIT_DIRS[split])
    assert names == CLASS_NAMES, f"{split} classes {names} differ from train {CLASS_NAMES}"

print("Classes:", ALL_NAMES)
for split, s in samples.items():
    print(f"{split:5}: {len(s):5} images, {sum(len(x.labels) for x in s):6} objects")

## 4. Khám phá dữ liệu

Cần kiểm tra trước khi train:
- **Cân bằng class** giữa các tập.
- **Kích thước vật sau khi thu nhỏ về 160×160.** Vật < 8 px rất khó nhận diện.
- **Rò rỉ dữ liệu (data leakage):** Roboflow tạo 3 bản augment cho mỗi ảnh gốc. Nếu các bản của cùng một ảnh nằm ở cả train và test thì kết quả test bị "đẹp giả".

In [ ]:
def describe_split(name: str, split_samples: list[Sample]) -> np.ndarray:
    # Print class counts / image sizes and return object sizes in model-input pixels.
    counts = np.bincount(np.concatenate([s.labels for s in split_samples]), minlength=NUM_CLASSES + 1)
    sizes = {}
    for s in split_samples:
        sizes[(s.width, s.height)] = sizes.get((s.width, s.height), 0) + 1
    obj_px = np.concatenate([
        np.maximum(s.boxes[:, 2] / s.width, s.boxes[:, 3] / s.height) * IMG_SIZE for s in split_samples
    ])
    per_class = ", ".join(f"{CLASS_NAMES[i - 1]}={counts[i]}" for i in range(1, NUM_CLASSES + 1))
    print(f"{name:5} | {per_class} | image sizes {sizes} | objects < 8 px: {np.mean(obj_px < 8) * 100:.1f}%")
    return obj_px


obj_sizes = {split: describe_split(split, s) for split, s in samples.items()}

plt.figure(figsize=(7, 3))
plt.hist(obj_sizes["train"], bins=40, color="tab:blue")
plt.axvline(8, color="red", linestyle="--", label="8 px")
plt.axvline(GRID_STRIDE, color="gray", linestyle=":", label="1 grid cell")
plt.xlabel(f"object size (longest side, px) at {IMG_SIZE}x{IMG_SIZE}")
plt.ylabel("count")
plt.legend()
plt.title("Object size after resize (train)")
plt.show()

# Leakage check: Roboflow names augmented copies "<source>.rf.<hash>.jpg"
source_ids = {split: {s.path.name.split(".rf.")[0] for s in ss} for split, ss in samples.items()}
print("Shared source images train/val :", len(source_ids["train"] & source_ids["val"]))
print("Shared source images train/test:", len(source_ids["train"] & source_ids["test"]))

> **Đọc kết quả:** nếu số ảnh nguồn dùng chung bằng 0 thì không có bản augment nào bị rò sang val/test.
> **Tuy nhiên**, Roboflow chia ngẫu nhiên theo từng ảnh, nên các **frame liền nhau của cùng một buổi chụp** vẫn có thể nằm ở cả train lẫn test.
> Vì vậy kết quả test ở đây có thể **lạc quan hơn thực tế**. Kế hoạch dự án yêu cầu chia theo buổi chụp:
> hãy giữ riêng **1 buổi chụp hoàn toàn mới** để kiểm tra cuối cùng trên board.

In [ ]:
def draw_samples(split_samples: list[Sample], n: int = 8) -> None:
    # Show random images with their ground-truth boxes.
    rng = np.random.default_rng(SEED)
    picks = rng.choice(len(split_samples), size=min(n, len(split_samples)), replace=False)
    colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(2, (len(picks) + 1) // 2, figsize=(16, 6))
    for ax, i in zip(axes.ravel(), picks):
        s = split_samples[i]
        with Image.open(s.path) as im:
            ax.imshow(im.convert("L"), cmap="gray")
        for (x, y, w, h), c in zip(s.boxes, s.labels):
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color=colors[c], linewidth=1.5))
            ax.text(x, y - 2, CLASS_NAMES[c - 1], color=colors[c], fontsize=8)
        ax.set_title(f"{s.width}x{s.height}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


draw_samples(samples["train"])

## 5. Tạo nhãn FOMO: từ bounding box sang lưới

FOMO **không học bounding box**. Mỗi box được đổi thành **tâm** của nó, và ô lưới chứa tâm đó được gán class của vật.
Các ô còn lại là **nền** (class 0). Nhãn của mỗi ảnh là một tensor one-hot `20×20×(C+1)`.

```
box (x, y, w, h) trên ảnh gốc ─► tâm (cx, cy) ─► ô (floor(cx/W·20), floor(cy/H·20)) = class
```

Hạn chế: 2 vật có tâm rơi vào **cùng một ô** sẽ chỉ còn 1 nhãn. Cell bên dưới đếm xem chuyện đó xảy ra bao nhiêu lần.

In [ ]:
def load_image(path: Path) -> np.ndarray:
    # Load as grayscale and squash-resize to IMG_SIZE x IMG_SIZE (must match firmware preprocessing).
    with Image.open(path) as im:
        im = im.convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        return np.asarray(im, dtype=np.uint8)


def gt_centers_grid(sample: Sample) -> np.ndarray:
    # Object centers in grid units (float), shape (N, 2) as (x, y).
    if len(sample.boxes) == 0:
        return np.zeros((0, 2), dtype=np.float32)
    cx = (sample.boxes[:, 0] + sample.boxes[:, 2] / 2) / sample.width * GRID_SIZE
    cy = (sample.boxes[:, 1] + sample.boxes[:, 3] / 2) / sample.height * GRID_SIZE
    return np.stack([cx, cy], axis=1).astype(np.float32)


def make_label_grid(sample: Sample) -> tuple[np.ndarray, int]:
    # Build the one-hot FOMO target. Returns (grid, number of objects lost to cell collisions).
    grid = np.zeros((GRID_SIZE, GRID_SIZE, NUM_CLASSES + 1), dtype=np.float32)
    grid[..., 0] = 1.0  # everything starts as background
    cells = np.clip(gt_centers_grid(sample).astype(np.int64), 0, GRID_SIZE - 1)
    collisions = 0
    for (cx, cy), cls in zip(cells, sample.labels):
        if grid[cy, cx, 0] == 0.0:
            collisions += 1  # cell already holds another object
        grid[cy, cx, :] = 0.0
        grid[cy, cx, cls] = 1.0
    return grid, collisions


def build_arrays(split_samples: list[Sample]) -> tuple[np.ndarray, np.ndarray]:
    # Load images (N, S, S, 1) uint8 and label grids (N, G, G, C+1) float32.
    images = np.zeros((len(split_samples), IMG_SIZE, IMG_SIZE, 1), dtype=np.uint8)
    labels = np.zeros((len(split_samples), GRID_SIZE, GRID_SIZE, NUM_CLASSES + 1), dtype=np.float32)
    lost = 0
    for i, s in enumerate(split_samples):
        images[i, ..., 0] = load_image(s.path)
        labels[i], n_lost = make_label_grid(s)
        lost += n_lost
    total = sum(len(s.labels) for s in split_samples)
    print(f"  {len(split_samples)} images | objects merged by grid collisions: {lost}/{total} ({lost / max(total, 1) * 100:.2f}%)")
    return images, labels


arrays = {}
for split, s in samples.items():
    print(split)
    arrays[split] = build_arrays(s)
X_train, Y_train = arrays["train"]
X_val, Y_val = arrays["val"]
X_test, Y_test = arrays["test"]
print("X_train", X_train.shape, X_train.dtype, "| Y_train", Y_train.shape)

## 6. Augmentation (chỉ áp dụng cho train)

Roboflow đã augment sẵn (độ sáng ±18%, blur, nhiễu). Notebook thêm augmentation **ngẫu nhiên mỗi epoch**, bám theo điều kiện thật của mô hình:
- **Lật ngang:** xe/người đi theo cả 2 chiều. Ảnh lật thì **lưới nhãn cũng phải lật** theo.
- **Độ sáng mạnh (×0.35 → ×1.3):** đèn chạy ở mức 20% (nền) đến 100%, và có lúc đang giảm sáng dở dang.
- **Contrast và nhiễu Gaussian nhẹ:** cảm biến OV2640 bị nhiễu khi thiếu sáng.

Input của model là **pixel thô 0..255**. Việc chuẩn hoá về [-1, 1] nằm **bên trong model** (layer `Rescaling`),
nên firmware không phải tự chuẩn hoá (xem phần lượng tử hoá).

In [ ]:
def augment(image: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    # Random geometric + photometric augmentation. Geometric ops are applied to the label grid too.
    image = tf.cast(image, tf.float32)

    flip_lr = tf.random.uniform(()) < 0.5
    image = tf.where(flip_lr, tf.reverse(image, axis=[1]), image)
    label = tf.where(flip_lr, tf.reverse(label, axis=[1]), label)

    if USE_VERTICAL_FLIP:
        flip_ud = tf.random.uniform(()) < 0.5
        image = tf.where(flip_ud, tf.reverse(image, axis=[0]), image)
        label = tf.where(flip_ud, tf.reverse(label, axis=[0]), label)

    mean = tf.reduce_mean(image)
    contrast = tf.random.uniform((), 0.7, 1.3)
    gain = tf.random.uniform((), 0.35, 1.3)  # simulates 20%..100% lamp brightness
    image = ((image - mean) * contrast + mean) * gain
    noise_std = tf.random.uniform((), 0.0, 6.0)
    image = image + tf.random.normal(tf.shape(image), stddev=noise_std)
    return tf.clip_by_value(image, 0.0, 255.0), label


def to_float(image: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    return tf.cast(image, tf.float32), label


AUTOTUNE = tf.data.AUTOTUNE
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, Y_train))
    .shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, Y_val)).map(to_float).batch(BATCH_SIZE).prefetch(AUTOTUNE)

X_val_f = X_val.astype(np.float32)
X_test_f = X_test.astype(np.float32)

**Kiểm tra augmentation có giữ đúng nhãn không:** các ô viền màu là ô có nhãn. Sau khi lật ảnh, ô nhãn phải vẫn nằm đúng trên vật.

In [ ]:
def show_label_overlay(images: np.ndarray, labels: np.ndarray, n: int = 6) -> None:
    # Draw labelled grid cells on top of (augmented) images.
    colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax, img, lab in zip(axes, images[:n], labels[:n]):
        ax.imshow(img[..., 0], cmap="gray", vmin=0, vmax=255)
        cls_map = lab.argmax(-1)
        for gy, gx in zip(*np.nonzero(cls_map)):
            ax.add_patch(plt.Rectangle((gx * GRID_STRIDE, gy * GRID_STRIDE), GRID_STRIDE, GRID_STRIDE,
                                       fill=False, color=colors[cls_map[gy, gx]], linewidth=1.5))
        ax.axis("off")
    plt.tight_layout()
    plt.show()


aug_images, aug_labels = next(iter(train_ds))
show_label_overlay(aug_images.numpy(), aug_labels.numpy())

## 7. Model FOMO

**Backbone:** MobileNetV2 α=0.35 **cắt ở `block_6_expand_relu`**. Tại đó ảnh đã giảm 8 lần (160 → 20), feature đủ "hiểu" hình dạng vật
mà vẫn giữ được vị trí. Phần sau của MobileNetV2 bị bỏ, nên model nhẹ.

**Trọng số ImageNet cho ảnh grayscale:** trọng số gốc dành cho ảnh 3 kênh RGB.
Ảnh xám đưa vào dưới dạng RGB thì R = G = B, nên lớp conv đầu tính `Σ_c W_c·x = (Σ_c W_c)·x`.
Vì vậy chỉ cần **cộng kernel của 3 kênh lại thành 1 kênh**. Kết quả giống hệt mà tiết kiệm 3 lần tính toán ở lớp đầu.

**Head:** Conv 1×1 (32 filter, ReLU), rồi Conv 1×1 (C+1 filter, softmax). Mỗi ô lưới được phân loại độc lập thành nền / car / human.

In [ ]:
def build_gray_mobilenetv2_trunk() -> keras.Model:
    # MobileNetV2 trunk with 1-channel input, initialised from RGB ImageNet weights.
    gray = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 1), alpha=ALPHA, include_top=False, weights=None
    )
    try:
        rgb = keras.applications.MobileNetV2(
            input_shape=(IMG_SIZE, IMG_SIZE, 3), alpha=ALPHA, include_top=False, weights="imagenet"
        )
    except Exception as exc:  # keras.utils.get_file raises a bare Exception on download failure
        print(f"WARNING: ImageNet weights unavailable ({exc}).\n"
              "         Training from scratch — turn Internet ON in Kaggle settings for much better results.")
    else:
        copied = 0
        for layer in gray.layers:
            if not layer.get_weights():
                continue
            weights = rgb.get_layer(layer.name).get_weights()
            if layer.name == "Conv1":
                # Gray image == RGB image with R=G=B, so summing the kernel over input channels is exact.
                weights = [weights[0].sum(axis=2, keepdims=True)] + weights[1:]
            layer.set_weights(weights)
            copied += 1
        print(f"Loaded ImageNet weights into {copied} layers (Conv1 collapsed RGB -> gray).")
    return keras.Model(gray.input, gray.get_layer(CUT_LAYER).output, name="mobilenetv2_trunk")


def build_fomo_model(trunk: keras.Model) -> keras.Model:
    # Raw pixels (0..255) in, per-cell class probabilities (G, G, C+1) out.
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 1), name="image")
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="to_minus1_plus1")(inputs)
    x = trunk(x)
    x = keras.layers.Conv2D(HEAD_FILTERS, 1, activation="relu", name="head_conv")(x)
    outputs = keras.layers.Conv2D(NUM_CLASSES + 1, 1, activation="softmax", name="heatmap")(x)
    return keras.Model(inputs, outputs, name="fomo")


trunk = build_gray_mobilenetv2_trunk()
model = build_fomo_model(trunk)
model.summary()
assert model.output_shape[1:] == (GRID_SIZE, GRID_SIZE, NUM_CLASSES + 1), model.output_shape

## 8. Loss: cross-entropy có trọng số

Mỗi ảnh có 400 ô nhưng chỉ khoảng 3 ô chứa vật. Nếu dùng cross-entropy thường, model đoán **toàn bộ là nền** đã đạt 99% "độ chính xác"
mà không phát hiện được gì. Vì vậy loss của ô có vật được **nhân `OBJECT_WEIGHT` (=100)**, giống FOMO của Edge Impulse.

- Tăng trọng số lên thì recall tăng nhưng dự đoán sai (false positive) cũng nhiều hơn.
- Giảm trọng số xuống thì ngược lại. Ngưỡng score ở phần 10 dùng để cân lại giữa hai bên.

In [ ]:
def make_weighted_fomo_loss(object_weight: float):
    # Per-cell categorical cross-entropy, object cells weighted by `object_weight`.
    def fomo_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        cross_entropy = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)   # (B, G, G)
        is_object = 1.0 - y_true[..., 0]
        weights = 1.0 + (object_weight - 1.0) * is_object
        return cross_entropy * weights
    return fomo_loss


fomo_loss = make_weighted_fomo_loss(OBJECT_WEIGHT)

## 9. Giải mã heatmap và metric theo tâm (Precision / Recall / F1)

**Giải mã (firmware cũng làm y hệt):**
1. Với mỗi class, đánh dấu các ô có xác suất ≥ ngưỡng.
2. Các ô đánh dấu **liền nhau** (4 hướng) được gom thành **một vật**. Tâm vật là trung bình toạ độ các ô, score là giá trị lớn nhất.

**Chấm điểm:** dự đoán là **TP** nếu cùng class và cách tâm thật ≤ `MATCH_DIST_CELLS` ô (mỗi tâm thật chỉ được ghép 1 lần).
Dự đoán thừa là **FP**, vật bị bỏ sót là **FN**. Tâm thật lấy từ **box gốc**, nên nếu 2 vật chung một ô thì sẽ bị tính FN, phản ánh đúng hạn chế của FOMO.

**Ma trận nhầm lẫn:** ghép không quan tâm class, để thấy model **nhầm người với xe** hay **bỏ sót / báo nhầm nền**.

In [ ]:
@dataclass(frozen=True)
class Detection:
    cls: int      # 1..C
    x: float      # centroid, grid units
    y: float
    score: float


def decode_heatmap(prob: np.ndarray, threshold: float) -> list[Detection]:
    # Threshold each class map and merge 4-connected cells into one detection each.
    detections = []
    for cls in range(1, prob.shape[-1]):
        mask = prob[..., cls] >= threshold
        if not mask.any():
            continue
        blobs, n_blobs = ndimage.label(mask)
        for blob_id in range(1, n_blobs + 1):
            ys, xs = np.nonzero(blobs == blob_id)
            detections.append(Detection(cls, xs.mean() + 0.5, ys.mean() + 0.5, float(prob[ys, xs, cls].max())))
    return detections


def greedy_match(pred_xy: np.ndarray, pred_score: np.ndarray, gt_xy: np.ndarray,
                 max_dist: float) -> list[tuple[int, int]]:
    # Match predictions (highest score first) to the nearest unused ground truth within max_dist.
    matches, used_gt = [], np.zeros(len(gt_xy), dtype=bool)
    if len(gt_xy) == 0:
        return matches
    for p in np.argsort(-pred_score):
        dist = np.linalg.norm(gt_xy - pred_xy[p], axis=1)
        dist[used_gt] = np.inf
        g = int(np.argmin(dist))
        if dist[g] <= max_dist:
            matches.append((int(p), g))
            used_gt[g] = True
    return matches


def evaluate_predictions(probs: np.ndarray, split_samples: list[Sample], threshold: float,
                         max_dist: float = MATCH_DIST_CELLS) -> dict:
    # Centroid-based precision/recall/F1 per class plus a (C+1)x(C+1) confusion matrix (row = truth).
    n = NUM_CLASSES + 1
    tp, fp, fn = np.zeros(n, int), np.zeros(n, int), np.zeros(n, int)
    confusion = np.zeros((n, n), int)
    for prob, sample in zip(probs, split_samples):
        dets = decode_heatmap(prob, threshold)
        p_xy = np.array([[d.x, d.y] for d in dets], dtype=np.float32).reshape(-1, 2)
        p_cls = np.array([d.cls for d in dets], dtype=int)
        p_score = np.array([d.score for d in dets], dtype=np.float32)
        g_xy, g_cls = gt_centers_grid(sample), sample.labels

        for cls in range(1, n):
            pm, gm = p_cls == cls, g_cls == cls
            n_match = len(greedy_match(p_xy[pm], p_score[pm], g_xy[gm], max_dist))
            tp[cls] += n_match
            fp[cls] += int(pm.sum()) - n_match
            fn[cls] += int(gm.sum()) - n_match

        matches = greedy_match(p_xy, p_score, g_xy, max_dist)  # class-agnostic, for confusion
        for p, g in matches:
            confusion[g_cls[g], p_cls[p]] += 1
        matched_p, matched_g = {p for p, _ in matches}, {g for _, g in matches}
        for g in set(range(len(g_cls))) - matched_g:
            confusion[g_cls[g], 0] += 1
        for p in set(range(len(p_cls))) - matched_p:
            confusion[0, p_cls[p]] += 1

    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / np.maximum(tp + fn, 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-9)
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn,
            "confusion": confusion, "macro_f1": float(f1[1:].mean()), "threshold": threshold}


def print_report(result: dict, title: str) -> None:
    print(f"{title}  (threshold={result['threshold']:.2f})")
    print(f"  {'class':<10}{'P':>7}{'R':>7}{'F1':>7}{'TP':>7}{'FP':>7}{'FN':>7}")
    for c in range(1, NUM_CLASSES + 1):
        print(f"  {CLASS_NAMES[c - 1]:<10}{result['precision'][c]:7.3f}{result['recall'][c]:7.3f}"
              f"{result['f1'][c]:7.3f}{result['tp'][c]:7d}{result['fp'][c]:7d}{result['fn'][c]:7d}")
    print(f"  macro F1 = {result['macro_f1']:.3f}")


# Sanity check: feeding the ground-truth grids as "predictions" gives the best F1 FOMO can reach.
# Misses come from (1) two objects in the same cell and (2) objects in ADJACENT cells being merged
# into one blob by decode_heatmap (e.g. people walking side by side).
print_report(evaluate_predictions(Y_val, samples["val"], 0.5), "Sanity check — labels as predictions (val)")

> **Đọc sanity check:** đây là **trần F1** của cách giải mã FOMO trên dataset này (đo được khoảng 0.98, recall `human` ≈ 0.95).
> Phần bị mất chủ yếu do **người đứng sát nhau** rơi vào các ô kề nhau và bị gom thành 1 vật.
> Với bài toán chiếu sáng thì không sao, vì 2 người sát nhau luôn cùng zone. Khi so sánh model, hãy so với trần này chứ không phải với 1.0.

## 10. Train

Train theo **2 giai đoạn transfer learning**:
1. **Đóng băng backbone** (`EPOCHS_FROZEN` epoch): chỉ head học. Tránh việc gradient lớn của head (đang khởi tạo ngẫu nhiên) phá trọng số ImageNet.
2. **Mở băng toàn bộ** với learning rate nhỏ hơn: tinh chỉnh cả backbone cho ảnh xám trên mô hình thu nhỏ.

Thước đo chính là **val macro F1** (theo tâm), không phải loss. Mỗi epoch, callback tính F1 trên val và lưu trọng số tốt nhất.

In [ ]:
class ValF1Checkpoint(keras.callbacks.Callback):
    # Computes centroid F1 on the validation set every epoch and keeps the best weights.

    def __init__(self, x_val: np.ndarray, val_samples: list[Sample], path: Path, threshold: float):
        super().__init__()
        self.x_val, self.val_samples, self.path, self.threshold = x_val, val_samples, path, threshold
        self.best_f1 = -1.0
        self.f1_history: list[float] = []

    def on_epoch_end(self, epoch: int, logs: dict | None = None) -> None:
        probs = self.model.predict(self.x_val, batch_size=64, verbose=0)
        f1 = evaluate_predictions(probs, self.val_samples, self.threshold)["macro_f1"]
        self.f1_history.append(f1)
        marker = ""
        if f1 > self.best_f1:
            self.best_f1 = f1
            self.model.save_weights(str(self.path))
            marker = "  <- best, saved"
        print(f"  val macro F1 = {f1:.4f}{marker}")


BEST_WEIGHTS = OUTPUT_DIR / "fomo_best.weights.h5"
f1_ckpt = ValF1Checkpoint(X_val_f, samples["val"], BEST_WEIGHTS, DEFAULT_THRESHOLD)

# Phase 1 — frozen backbone
trunk.trainable = False
model.compile(optimizer=keras.optimizers.Adam(LR_FROZEN), loss=fomo_loss)
hist_frozen = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN, callbacks=[f1_ckpt], verbose=2)

In [ ]:
# Phase 2 — fine-tune everything (recompile is required after changing `trainable`)
trunk.trainable = True
model.compile(optimizer=keras.optimizers.Adam(LR_FINETUNE), loss=fomo_loss)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5, verbose=1)
hist_finetune = model.fit(
    train_ds, validation_data=val_ds,
    initial_epoch=EPOCHS_FROZEN, epochs=EPOCHS_FROZEN + EPOCHS_FINETUNE,
    callbacks=[f1_ckpt, reduce_lr], verbose=2,
)
print("Best val macro F1 during training:", round(f1_ckpt.best_f1, 4))

In [ ]:
loss = hist_frozen.history["loss"] + hist_finetune.history["loss"]
val_loss = hist_frozen.history["val_loss"] + hist_finetune.history["val_loss"]
epochs = np.arange(1, len(loss) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(epochs, loss, label="train loss")
ax1.plot(epochs, val_loss, label="val loss")
ax1.axvline(EPOCHS_FROZEN + 0.5, color="gray", linestyle=":", label="unfreeze backbone")
ax1.set_xlabel("epoch")
ax1.set_title("Weighted FOMO loss")
ax1.legend()
ax2.plot(epochs, f1_ckpt.f1_history, color="tab:green")
ax2.axvline(EPOCHS_FROZEN + 0.5, color="gray", linestyle=":")
ax2.set_xlabel("epoch")
ax2.set_title(f"val macro F1 (threshold {DEFAULT_THRESHOLD})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=120)
plt.show()

model.load_weights(str(BEST_WEIGHTS))
print("Restored best weights from", BEST_WEIGHTS)

## 11. Chọn ngưỡng score trên **val**, rồi đánh giá trên **test**

Ngưỡng score (`FOMO_SCORE_MIN` trong firmware) cân bằng giữa precision và recall:
- Ngưỡng thấp: ít bỏ sót vật, nhưng dễ bật đèn nhầm.
- Ngưỡng cao: ít báo nhầm, nhưng có thể bỏ sót người/xe.

**Quy tắc:** chỉ chọn ngưỡng trên **val**. **Test** chỉ dùng **một lần** để báo cáo, nếu không con số test sẽ không còn khách quan.

In [ ]:
val_probs = model.predict(X_val_f, batch_size=64, verbose=0)
sweep = [evaluate_predictions(val_probs, samples["val"], t) for t in THRESHOLDS]
best = max(sweep, key=lambda r: r["macro_f1"])
BEST_THRESHOLD = float(best["threshold"])

plt.figure(figsize=(7, 4))
plt.plot(THRESHOLDS, [r["precision"][1:].mean() for r in sweep], label="macro precision")
plt.plot(THRESHOLDS, [r["recall"][1:].mean() for r in sweep], label="macro recall")
plt.plot(THRESHOLDS, [r["macro_f1"] for r in sweep], label="macro F1", linewidth=2)
plt.axvline(BEST_THRESHOLD, color="gray", linestyle=":", label=f"best = {BEST_THRESHOLD:.2f}")
plt.xlabel("score threshold")
plt.title("Threshold sweep on validation set")
plt.legend()
plt.savefig(OUTPUT_DIR / "threshold_sweep.png", dpi=120)
plt.show()
print_report(best, "VALIDATION — float model")

In [ ]:
def plot_confusion(confusion: np.ndarray, title: str) -> None:
    plt.figure(figsize=(4.5, 4))
    plt.imshow(confusion, cmap="Blues")
    for i in range(confusion.shape[0]):
        for j in range(confusion.shape[1]):
            plt.text(j, i, confusion[i, j], ha="center", va="center",
                     color="white" if confusion[i, j] > confusion.max() / 2 else "black")
    plt.xticks(range(len(ALL_NAMES)), ALL_NAMES, rotation=30)
    plt.yticks(range(len(ALL_NAMES)), ALL_NAMES)
    plt.xlabel("predicted")
    plt.ylabel("ground truth")
    plt.title(title)
    plt.tight_layout()
    plt.show()


test_probs_float = model.predict(X_test_f, batch_size=64, verbose=0)
test_float = evaluate_predictions(test_probs_float, samples["test"], BEST_THRESHOLD)
print_report(test_float, "TEST — float model")
plot_confusion(test_float["confusion"], "Test confusion (float)")

In [ ]:
def show_predictions(images: np.ndarray, probs: np.ndarray, split_samples: list[Sample],
                     threshold: float, n: int = 8) -> None:
    # Green circles = ground-truth centers, coloured crosses = predicted centroids.
    colors = plt.cm.tab10.colors
    rng = np.random.default_rng(SEED + 1)
    picks = rng.choice(len(images), size=min(n, len(images)), replace=False)
    fig, axes = plt.subplots(2, (len(picks) + 1) // 2, figsize=(16, 8))
    for ax, i in zip(axes.ravel(), picks):
        ax.imshow(images[i, ..., 0], cmap="gray", vmin=0, vmax=255)
        for gx, gy in gt_centers_grid(split_samples[i]) * GRID_STRIDE:
            ax.add_patch(plt.Circle((gx, gy), 6, fill=False, color="lime", linewidth=1.5))
        for d in decode_heatmap(probs[i], threshold):
            ax.plot(d.x * GRID_STRIDE, d.y * GRID_STRIDE, "x", color=colors[d.cls], markersize=10, mew=2)
            ax.text(d.x * GRID_STRIDE + 4, d.y * GRID_STRIDE - 4, f"{CLASS_NAMES[d.cls - 1]} {d.score:.2f}",
                    color=colors[d.cls], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_predictions(X_test, test_probs_float, samples["test"], BEST_THRESHOLD)

## 12. Lượng tử hoá int8 (full-integer)

**Vì sao cần int8:** trên ESP32-S3, phép tính int8 nhanh hơn nhiều so với float (ESP-NN có lệnh SIMD cho int8).
Model cũng nhỏ đi khoảng 4 lần. Converter cần một **representative dataset** để đo dải giá trị của từng tensor.
Dataset này được lấy **chỉ từ train**, để val/test vẫn là dữ liệu "chưa thấy" khi đánh giá model int8.

**Input int8:** vì model nhận pixel 0..255, converter sẽ chọn `scale ≈ 1`, `zero_point = -128`.
Khi đó firmware chỉ cần làm `input_int8 = pixel - 128`. Cell bên dưới in ra giá trị thật để kiểm tra.

In [ ]:
calib_rng = np.random.default_rng(SEED)
calib_idx = calib_rng.choice(len(X_train), size=min(NUM_CALIB_IMAGES, len(X_train)), replace=False)


def representative_dataset():
    for i in calib_idx:
        yield [X_train[i:i + 1].astype(np.float32)]


def configure_int8(converter: tf.lite.TFLiteConverter) -> tf.lite.TFLiteConverter:
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    return converter


def convert_to_int8_tflite(keras_model: keras.Model) -> bytes:
    # Full-integer conversion; falls back to the SavedModel path if from_keras_model fails (Keras 3).
    try:
        return configure_int8(tf.lite.TFLiteConverter.from_keras_model(keras_model)).convert()
    except Exception as exc:
        print(f"from_keras_model failed ({type(exc).__name__}: {exc}); retrying via SavedModel export")
    saved_dir = OUTPUT_DIR / "saved_model"
    if hasattr(keras_model, "export"):
        keras_model.export(str(saved_dir))          # Keras 3
    else:
        tf.saved_model.save(keras_model, str(saved_dir))
    return configure_int8(tf.lite.TFLiteConverter.from_saved_model(str(saved_dir))).convert()


tflite_int8 = convert_to_int8_tflite(model)
TFLITE_PATH = OUTPUT_DIR / "fomo_int8.tflite"
TFLITE_PATH.write_bytes(tflite_int8)
print(f"Saved {TFLITE_PATH} ({len(tflite_int8) / 1024:.1f} KB)")

In [ ]:
def make_interpreter(model_bytes: bytes) -> tf.lite.Interpreter:
    interpreter = tf.lite.Interpreter(model_content=model_bytes)
    interpreter.allocate_tensors()
    return interpreter


def predict_int8(model_bytes: bytes, images_uint8: np.ndarray) -> np.ndarray:
    # Run the int8 model image by image (like the MCU) and return de-quantized probabilities.
    interpreter = make_interpreter(model_bytes)
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    in_scale, in_zp = inp["quantization"]
    out_scale, out_zp = out["quantization"]
    results = np.zeros((len(images_uint8),) + tuple(out["shape"][1:]), dtype=np.float32)
    for i, img in enumerate(images_uint8):
        q = np.clip(np.round(img.astype(np.float32) / in_scale + in_zp), -128, 127).astype(np.int8)
        interpreter.set_tensor(inp["index"], q[np.newaxis])
        interpreter.invoke()
        results[i] = (interpreter.get_tensor(out["index"])[0].astype(np.float32) - out_zp) * out_scale
    return results


_interp = make_interpreter(tflite_int8)
IN_DETAILS, OUT_DETAILS = _interp.get_input_details()[0], _interp.get_output_details()[0]
IN_SCALE, IN_ZP = IN_DETAILS["quantization"]
OUT_SCALE, OUT_ZP = OUT_DETAILS["quantization"]
print("Input :", IN_DETAILS["shape"], IN_DETAILS["dtype"].__name__, f"scale={IN_SCALE:.6f} zero_point={IN_ZP}")
print("Output:", OUT_DETAILS["shape"], OUT_DETAILS["dtype"].__name__, f"scale={OUT_SCALE:.6f} zero_point={OUT_ZP}")
print(f"Firmware input formula: int8 = round(pixel / {IN_SCALE:.6f}) + ({IN_ZP})")

### So sánh float và int8 trên test (cùng ngưỡng đã chọn trên val)

Model int8 **chỉ được coi là xong** khi đã so với model float và hiểu được mức chênh lệch.
Thường F1 giảm < 1–2 điểm. Nếu giảm nhiều hơn, hãy xem lại representative dataset (đủ đa dạng ánh sáng chưa?).

In [ ]:
test_probs_int8 = predict_int8(tflite_int8, X_test)
test_int8 = evaluate_predictions(test_probs_int8, samples["test"], BEST_THRESHOLD)
print_report(test_int8, "TEST — int8 TFLite model")

print("\nF1 change (int8 - float):")
for c in range(1, NUM_CLASSES + 1):
    print(f"  {CLASS_NAMES[c - 1]:<10}{test_int8['f1'][c] - test_float['f1'][c]:+.4f}")
print(f"  {'macro':<10}{test_int8['macro_f1'] - test_float['macro_f1']:+.4f}")
plot_confusion(test_int8["confusion"], "Test confusion (int8)")

## 13. Tài nguyên trên board: Flash, RAM, danh sách op

- **Flash** = firmware + runtime TFLite Micro + kernel các op + **file model**. Notebook chỉ đo được phần model.
  Ba phần còn lại xem bằng `idf.py size-components` sau khi build.
- **RAM (tensor arena):** ước lượng **cận dưới** bằng cách tìm op có tổng input + output activation lớn nhất.
  Con số thật phải đo trên board bằng `interpreter.arena_used_bytes()`. Nếu vượt SRAM nội (512 KB), đặt arena trong **PSRAM**.
- **Danh sách op** dùng cho `MicroMutableOpResolver`, chỉ đăng ký đúng các op cần, không dùng `AllOpsResolver` (tốn Flash).

In [ ]:
OP_TO_RESOLVER = {
    "CONV_2D": "AddConv2D", "DEPTHWISE_CONV_2D": "AddDepthwiseConv2D", "ADD": "AddAdd", "MUL": "AddMul",
    "SOFTMAX": "AddSoftmax", "PAD": "AddPad", "RESHAPE": "AddReshape", "QUANTIZE": "AddQuantize",
    "DEQUANTIZE": "AddDequantize", "RELU": "AddRelu", "RELU6": "AddRelu6", "MEAN": "AddMean",
}


def analyze_tflite(model_bytes: bytes) -> tuple[list[str], int]:
    # Return the sorted list of op names and a lower-bound estimate of peak activation memory (bytes).
    interpreter = make_interpreter(model_bytes)
    ops = interpreter._get_ops_details()
    tensors = {t["index"]: t for t in interpreter.get_tensor_details()}
    activation_ids = {i for op in ops for i in op["outputs"]}
    activation_ids |= {d["index"] for d in interpreter.get_input_details()}

    def nbytes(idx: int) -> int:
        t = tensors[idx]
        return int(np.prod(t["shape"])) * np.dtype(t["dtype"]).itemsize

    peak = 0
    for op in ops:
        live = [i for i in list(op["inputs"]) + list(op["outputs"]) if i in activation_ids]
        peak = max(peak, sum(nbytes(i) for i in live))
    return sorted({op["op_name"] for op in ops}), peak


op_names, peak_activation = analyze_tflite(tflite_int8)
NUM_OPS = len(op_names)
print(f"Model file (Flash)            : {len(tflite_int8) / 1024:8.1f} KB")
print(f"Peak activation (lower bound) : {peak_activation / 1024:8.1f} KB  -> arena will be somewhat larger")
print("Ops used:", op_names)

resolver_lines = [f"static tflite::MicroMutableOpResolver<{NUM_OPS}> resolver;"]
for name in op_names:
    method = OP_TO_RESOLVER.get(name)
    resolver_lines.append(f"resolver.{method}();" if method else f"// TODO: register op {name} manually")
RESOLVER_SNIPPET = "\n".join(resolver_lines)
print("\nC++ op resolver for TFLite Micro:\n" + RESOLVER_SNIPPET)

## 14. Xuất file cho firmware (ESP-IDF + TFLite Micro)

Các file trong `fomo_output/`:

| File | Dùng để |
|---|---|
| `fomo_int8.tflite` | Model int8 (xem bằng Netron) |
| `fomo_model_data.h/.cc` | Model dạng mảng C (`alignas(16)`), nằm trong Flash, kèm các hằng số input/output/ngưỡng |
| `fomo_metadata.json` | Toàn bộ cấu hình + kết quả đánh giá (để ghi vào báo cáo) |
| `fomo_best.weights.h5`, `fomo_float.keras` | Model float để train tiếp / thử nghiệm |
| `*.png` | Biểu đồ cho báo cáo |

In [ ]:
def write_c_model(model_bytes: bytes, out_dir: Path) -> None:
    # Write fomo_model_data.h / .cc containing the model bytes and preprocessing constants.
    labels_c = ", ".join(f'"{n}"' for n in ALL_NAMES)
    header = "\n".join([
        "// Auto-generated by train_fomo_kaggle.ipynb - do not edit by hand.",
        "#pragma once",
        "#include <stdint.h>",
        "",
        f"#define FOMO_INPUT_WIDTH        {IMG_SIZE}",
        f"#define FOMO_INPUT_HEIGHT       {IMG_SIZE}",
        "#define FOMO_INPUT_CHANNELS     1   // grayscale",
        f"#define FOMO_GRID_WIDTH         {GRID_SIZE}",
        f"#define FOMO_GRID_HEIGHT        {GRID_SIZE}",
        f"#define FOMO_GRID_STRIDE        {GRID_STRIDE}   // input pixels per grid cell",
        f"#define FOMO_NUM_CLASSES        {NUM_CLASSES + 1}   // including background (index 0)",
        f"#define FOMO_INPUT_SCALE        {IN_SCALE:.9g}f",
        f"#define FOMO_INPUT_ZERO_POINT   ({IN_ZP})",
        f"#define FOMO_OUTPUT_SCALE       {OUT_SCALE:.9g}f",
        f"#define FOMO_OUTPUT_ZERO_POINT  ({OUT_ZP})",
        f"#define FOMO_SCORE_THRESHOLD    {BEST_THRESHOLD:.2f}f   // chosen on validation set",
        "",
        "// Preprocessing: crop ROI, squash-resize to FOMO_INPUT_WIDTH x FOMO_INPUT_HEIGHT (bilinear),",
        "// then input_int8 = round(pixel / FOMO_INPUT_SCALE) + FOMO_INPUT_ZERO_POINT.",
        "",
        "extern const unsigned char g_fomo_model[];",
        "extern const unsigned int g_fomo_model_len;",
        "extern const char *const g_fomo_labels[FOMO_NUM_CLASSES];",
        "",
    ])
    hex_rows = [
        "  " + ", ".join(f"0x{b:02x}" for b in model_bytes[i:i + 12]) + ","
        for i in range(0, len(model_bytes), 12)
    ]
    source = "\n".join([
        "// Auto-generated by train_fomo_kaggle.ipynb - do not edit by hand.",
        '#include "fomo_model_data.h"',
        "",
        f"const char *const g_fomo_labels[FOMO_NUM_CLASSES] = {{{labels_c}}};",
        "",
        "// 16-byte alignment is required by TFLite Micro for the flatbuffer.",
        "alignas(16) const unsigned char g_fomo_model[] = {",
        *hex_rows,
        "};",
        f"const unsigned int g_fomo_model_len = {len(model_bytes)};",
        "",
    ])
    (out_dir / "fomo_model_data.h").write_text(header, encoding="utf-8")
    (out_dir / "fomo_model_data.cc").write_text(source, encoding="utf-8")


def to_jsonable(result: dict) -> dict:
    return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in result.items()}


write_c_model(tflite_int8, OUTPUT_DIR)

metadata = {
    "classes": ALL_NAMES,
    "input": {"width": IMG_SIZE, "height": IMG_SIZE, "channels": 1, "resize": "squash, bilinear",
              "scale": float(IN_SCALE), "zero_point": int(IN_ZP)},
    "output": {"grid": GRID_SIZE, "stride": GRID_STRIDE, "scale": float(OUT_SCALE), "zero_point": int(OUT_ZP)},
    "model": {"backbone": f"MobileNetV2 alpha={ALPHA}", "cut_layer": CUT_LAYER, "head_filters": HEAD_FILTERS,
              "tflite_bytes": len(tflite_int8), "peak_activation_bytes_lower_bound": int(peak_activation),
              "ops": op_names},
    "training": {"epochs_frozen": EPOCHS_FROZEN, "epochs_finetune": EPOCHS_FINETUNE, "batch_size": BATCH_SIZE,
                 "object_weight": OBJECT_WEIGHT, "best_val_f1_at_default_threshold": f1_ckpt.best_f1},
    "evaluation": {"match_dist_cells": MATCH_DIST_CELLS, "score_threshold": BEST_THRESHOLD,
                   "val_float": to_jsonable(best), "test_float": to_jsonable(test_float),
                   "test_int8": to_jsonable(test_int8)},
    "dataset_root": str(DATA_ROOT),
}
(OUTPUT_DIR / "fomo_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

try:
    model.save(str(OUTPUT_DIR / "fomo_float.keras"))
except Exception as exc:  # saving a model compiled with a custom loss can fail on some Keras versions
    print(f"WARNING: could not save full .keras model ({exc}); best weights are in {BEST_WEIGHTS}")

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Exported files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:<28}{p.stat().st_size / 1024:10.1f} KB")
print("Zip for download:", archive)

## 15. Bước tiếp theo trên firmware

1. Copy `fomo_model_data.h/.cc` vào một component (vd `firmware/components/vision/`), thêm `esp-tflite-micro` (đã có ESP-NN tăng tốc int8).
2. **Tiền xử lý giống hệt notebook:** frame GRAYSCALE → cắt ROI → resize bilinear (ép) về 160×160 → `int8 = pixel - 128` (kiểm tra lại với `FOMO_INPUT_SCALE/ZERO_POINT`).
3. **Hậu xử lý giống `decode_heatmap`:** lấy các ô có `score ≥ FOMO_SCORE_THRESHOLD` (đổi score về float bằng `OUTPUT_SCALE/ZERO_POINT`), gom các ô liền nhau, lấy tâm.
   Tâm x (đơn vị ô, 0..20) → zone = `x * 4 / 20`.
4. **Đo trên board:** `arena_used_bytes()` (RAM thật), thời gian `Invoke()` (mục tiêu ≥ 3–5 lần/giây), `idf.py size-components` (Flash của runtime + kernel).
5. **Kiểm tra trên 1 buổi chụp mới** chưa từng dùng để train (xem lưu ý rò rỉ dữ liệu ở phần 4).

**Nếu kết quả chưa tốt, có thể thử:**
- Recall thấp: giảm ngưỡng, tăng `OBJECT_WEIGHT`, kiểm tra vật quá nhỏ (cắt ROI chặt hơn).
- Precision thấp (báo nhầm): tăng ngưỡng, thêm ảnh negative (mô hình trống, tay người đẩy xe) vào dataset.
- Nhầm người với xe: xem ma trận nhầm lẫn, thêm ảnh của class bị nhầm.
- F1 int8 giảm nhiều: tăng `NUM_CALIB_IMAGES`, kiểm tra representative set có đủ ảnh tối/sáng.